# 05. Análise de overfitting por validação cruzada interna

Estudo obrigatório da Etapa 2, feito inteiramente na fase de desenvolvimento:

1. **lacuna treino-validação** de todos os sistemas (modelo do grupo padrão e com busca, LightGBM, XGBoost, CatBoost), a partir de `results/validation.csv` e `results/validation_train.csv` (`python -m src.pipeline.run_validation`);
2. **curvas de validação de complexidade** por CV interna no treino do fold externo 0 de um subconjunto declarado de datasets (um por regime de tamanho), para (a) um hiperparâmetro de complexidade do modelo tema e (b) `num_leaves` do LightGBM, com escolha pela regra de 1 erro padrão;
3. **erros por classe**: matriz de confusão e recall por classe nas predições de **validação interna** gravadas por `run_validation` em `results/preds_val/` (nunca predições de teste).

O notebook usa as funções que o grupo implementa em `src/pipeline/overfitting.py`, `src/pipeline/class_errors.py` e `src/reports/plots_overfitting.py`.

**Teste selado.** Este notebook pertence à fase de desenvolvimento: usa somente o TREINO de um fold externo oficial (`get_train_indices`) e mede o desempenho por validação cruzada INTERNA. O teste dos folds externos só é usado uma vez, em `python -m src.pipeline.final_test`, depois do congelamento (`python -m src.pipeline.freeze`).

**Validação é diagnóstico.** O treino de um fold externo contém o teste dos outros dois folds externos. Por isso, os resultados desta fase (`results/validation.csv`, estudos e curvas) não são base de decisões: os sistemas, o controle `group_tuned`, o wrapper, a configuração padrão, o espaço de busca e o orçamento são fixados a priori. Depois de `run_validation`, só são permitidas correções de falha de execução (status `error`, contrato do estimador), registradas no relatório; mudanças motivadas por desempenho não são permitidas. O resultado deste notebook não altera o espaço de busca nem o `protocolo.lock`.

In [ ]:
import sys
from pathlib import Path

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd

from data.load_tabarena import get_train_indices, load_task, read_selection, summarize
from src.models.baselines import build_lightgbm
from src.models.group_model import build_group_model
from src.pipeline.class_errors import confusion_table, recall_per_class
from src.pipeline.cv_grid import inner_cv_splits, stratified_subsample
from src.pipeline.evaluate import HIGHER_IS_BETTER
from src.pipeline.overfitting import generalization_gap, select_complexity, validation_curve
from src.pipeline.results import load_validation
from src.reports.plots_overfitting import (
    plot_confusion,
    plot_generalization_gap,
    plot_validation_curve,
)

RESULTS = REPO / 'results'
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
METRIC = 'auc_ovo'
HIB = HIGHER_IS_BETTER[METRIC]
OUTER_FOLD = 0   # fold externo cujo TREINO é usado nas curvas (enunciado: fold externo 0)
MAX_SAMPLES = 10_000
PREDS_VAL = RESULTS / 'preds_val'   # predições de validação interna (run_validation)
SYSTEMS = ['group_default', 'group_tuned', 'lightgbm', 'xgboost', 'catboost']

## 1. Subconjunto declarado de datasets

Declare o subconjunto ANTES de olhar as curvas e registre-o no relatório. Sugestão: o primeiro dataset da seleção em cada regime de tamanho.

In [ ]:
sel = summarize(read_selection())
STUDY_TASKS = sel.groupby('regime_size', sort=False)['task_id'].first().to_dict()
STUDY_TASKS

## 2. Lacuna treino-validação de todos os sistemas

In [ ]:
val = load_validation(RESULTS / 'validation.csv')
val_train = load_validation(RESULTS / 'validation.csv', train=True)
gap = generalization_gap(val, val_train, METRIC, higher_is_better=HIB)
gap

In [ ]:
plot_generalization_gap(gap, output=FIGURES / f'lacuna_{METRIC}.png');

In [ ]:
# Lacuna por regime de tamanho (mesma regra, só nos datasets de cada regime).
for regime, ids in sel.groupby('regime_size')['task_id']:
    sub_v, sub_t = val[val['task_id'].isin(ids)], val_train[val_train['task_id'].isin(ids)]
    print(f'\n=== {regime} ===')
    print(generalization_gap(sub_v, sub_t, METRIC, higher_is_better=HIB).to_string(index=False))

As linhas de `group_tuned` têm `optimistic = True`: os hiperparâmetros foram escolhidos nas mesmas partições internas, de modo que a validação desse sistema é otimista e a lacuna tende a ser subestimada.

## 3. Curvas de validação de complexidade

Para cada dataset do subconjunto, apenas o treino do fold externo 0 é carregado; as partições internas são as mesmas da busca (`inner_cv_splits`).

In [ ]:
def development_data(task_id):
    ds = load_task(task_id)
    train_idx = get_train_indices(task_id, folds=[OUTER_FOLD])[0]
    X, y = stratified_subsample(ds.X.iloc[train_idx], ds.y[train_idx], MAX_SAMPLES, seed=42)
    return ds.name, X, y, inner_cv_splits(y, n_folds=3, seed=42)

### 3a. Hiperparâmetro de complexidade do modelo tema

Escolha um hiperparâmetro que controle a complexidade (por exemplo, número de camadas, largura, número de vizinhos, regularização) e declare a grade e o sentido de simplicidade (`SIMPLER_IS_LOWER`).

In [ ]:
GROUP_PARAM = None       # ex.: 'n_blocks' (nome aceito por build_group_model)
GROUP_VALUES = []        # ex.: [1, 2, 3, 4]
SIMPLER_IS_LOWER = True  # False quando valores maiores são mais simples (ex.: regularização)

group_curves = {}
if GROUP_PARAM is None:
    print('Defina GROUP_PARAM, GROUP_VALUES e SIMPLER_IS_LOWER.')
else:
    for regime, task_id in STUDY_TASKS.items():
        name, X, y, folds = development_data(task_id)
        curve = validation_curve(build_group_model, GROUP_PARAM, GROUP_VALUES, X, y, folds)
        group_curves[name] = curve
        best = select_complexity(curve, METRIC, 'best', SIMPLER_IS_LOWER, HIB)
        one_se = select_complexity(curve, METRIC, '1se', SIMPLER_IS_LOWER, HIB)
        print(f'{regime} | {name}: melhor média = {best}; regra de 1 EP = {one_se}')
        plot_validation_curve(curve, METRIC, output=FIGURES / f'curva_grupo_{task_id}.png')

### 3b. Hiperparâmetro de complexidade de um baseline GBDT: `num_leaves` do LightGBM

In [ ]:
LGBM_VALUES = [2, 4, 8, 16, 32, 64, 128]
lgbm_curves = {}
for regime, task_id in STUDY_TASKS.items():
    name, X, y, folds = development_data(task_id)
    curve = validation_curve(build_lightgbm, 'num_leaves', LGBM_VALUES, X, y, folds)
    lgbm_curves[name] = curve
    best = select_complexity(curve, METRIC, 'best', True, HIB)
    one_se = select_complexity(curve, METRIC, '1se', True, HIB)
    print(f'{regime} | {name}: melhor média = {best}; regra de 1 EP = {one_se}')
    plot_validation_curve(curve, METRIC, log_x=True, output=FIGURES / f'curva_lgbm_{task_id}.png')

## 4. Erros por classe nas predições de validação

Para cada dataset do subconjunto declarado, as predições de validação dos 3 folds internos do fold externo `OUTER_FOLD` são somadas numa matriz de confusão por sistema (cada instância do treino do fold externo é validada uma vez). A matriz normalizada por linha tem o recall de cada classe na diagonal; a média geométrica desses recalls é o G-Mean. Nenhuma predição de teste é usada.

In [ ]:
def validation_confusion(task_id, model, fold=OUTER_FOLD):
    """Matriz de confusão (contagens) e recall por classe somando os folds internos."""
    tables = []
    for path in sorted(PREDS_VAL.glob(f'{task_id}_fold{fold}_inner*_{model}.npz')):
        z = np.load(path)
        tables.append(confusion_table(z['y_true'], z['y_proba'], z['classes']))
    if not tables:
        return None
    counts = tables[0]
    for t in tables[1:]:
        counts = counts.add(t, fill_value=0).astype(int)
    support = counts.sum(axis=1)
    recall = pd.DataFrame({'class': counts.index, 'support': support.to_numpy(),
                           'recall': (np.diag(counts) / support.where(support > 0)).to_numpy()})
    return counts, recall


recalls = []
for regime, task_id in STUDY_TASKS.items():
    for model in SYSTEMS:
        out = validation_confusion(task_id, model)
        if out is None:
            print(f'Sem predições de validação para {task_id} / {model}: execute run_validation.')
            continue
        counts, recall = out
        recalls.append(recall.assign(regime=regime, task_id=task_id, model=model))
        rates = counts.div(counts.sum(axis=1).where(lambda s: s > 0), axis=0).fillna(0.0)
        plot_confusion(rates, output=FIGURES / f'confusao_{task_id}_{model}.png')
if recalls:
    display(pd.concat(recalls).pivot_table(index=['task_id', 'class'], columns='model', values='recall'))

In [ ]:
# Conferência das funções do grupo com uma única partição interna (recall_per_class).
task_id = next(iter(STUDY_TASKS.values()))
files = sorted(PREDS_VAL.glob(f'{task_id}_fold{OUTER_FOLD}_inner0_group_tuned.npz'))
if files:
    z = np.load(files[0])
    display(recall_per_class(z['y_true'], z['y_proba'], z['classes']))

## 5. Perguntas

**Pergunta 1.** Quais sistemas apresentam a maior lacuna treino-validação? A lacuna é maior nos datasets pequenos? Relacione com o compromisso entre viés e variância (Hastie, Tibshirani e Friedman, 2009, Seções 2.9 e 7.3).

**Pergunta 2.** Nas curvas de complexidade, a partir de que valor a validação deixa de melhorar enquanto o treino continua melhorando? Esse ponto muda entre regimes de tamanho?

**Pergunta 3.** A regra de 1 erro padrão escolhe um valor diferente da melhor média? Que vantagem tem escolher o modelo mais simples dentro de 1 EP (Breiman et al., 1984)? O valor escolhido está dentro do espaço de busca declarado em `search_space`?

**Pergunta 4.** Por que a validação de `group_tuned` é otimista? Como a CV aninhada (externa: folds oficiais; interna: busca) corrige esse viés na avaliação final (Cawley e Talbot, 2010; Varma e Simon, 2006)?

**Pergunta 5.** Por que nenhum destes estudos pode usar o teste dos folds externos? O que aconteceria com a estimativa final se a grade de `search_space` fosse revista depois de ver `results/test.csv`?

**Pergunta 6.** Quais classes concentram os erros de validação de cada sistema? As classes raras têm recall menor? Como isso se reflete no G-Mean e na AUC OvO? Os padrões de erro do modelo tema diferem dos baselines GBDT?

**Pergunta 7.** Por que a matriz de confusão é calculada nas predições de validação interna, e não nas de teste? Por que o resultado deste notebook não pode alterar o `protocolo.lock`?